In [1]:
import pandas as pd
from pathlib import Path
Dataset = Path("Dataset")
products = pd.read_csv(Dataset/'olist_products_dataset.csv')
translation = pd.read_csv(Dataset/'product_category_name_translation.csv')

In [2]:
products = pd.merge(products, translation, on='product_category_name', how='left')

In [3]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [4]:
products.drop(columns=['product_category_name'], inplace=True)

In [5]:
products.rename(columns={"product_category_name_english":"product_category"},inplace=True)

In [6]:
geo = pd.read_csv(Dataset/'olist_geolocation_dataset.csv')

In [7]:
geo_clean = geo.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'median',
    'geolocation_lng': 'median',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

In [8]:
payments = pd.read_csv(Dataset/'olist_order_payments_dataset.csv')

In [9]:
payments_clean = payments.groupby('order_id').agg({
    'payment_value': 'sum',
    'payment_installments': 'max'
}).reset_index()

In [10]:
sellers = pd.read_csv(Dataset/'olist_sellers_dataset.csv')

In [11]:
sellers['seller_city'] = sellers['seller_city'].str.lower().str.strip()

In [12]:
sellers['seller_city'].unique()

<ArrowStringArray>
[              'campinas',             'mogi guacu',         'rio de janeiro',
              'sao paulo',      'braganca paulista',                 'brejao',
              'penapolis',               'curitiba',               'anapolis',
              'itirapina',
 ...
          'varzea alegre',          'guaratingueta',                 'tambau',
                  'irati',          'riberao preto',   'aparecida de goiania',
           'bandeirantes', 'vitoria de santo antao',               'palotina',
                   'leme']
Length: 611, dtype: str

In [13]:
city_map = {
    'sao paulo': 'sao paulo',
    'são paulo': 'sao paulo',
    'saopaulo': 'sao paulo',
    'bragança paulista': 'braganca paulista',
    'belo horizonte': 'belo horizonte',
    'rio de janeiro': 'rio de janeiro'
}

In [14]:
sellers['seller_city'] = sellers['seller_city'].replace(city_map)

In [15]:
orders = pd.read_csv(Dataset/'olist_orders_dataset.csv')

In [16]:
date_cols = ['order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [17]:

orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

In [18]:
orders['is_late'] = orders['delivery_delay_days'] > 0

In [19]:
df = pd.merge(orders, payments_clean, on='order_id', how='left')

In [20]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,is_late,payment_value,payment_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,False,38.71,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,False,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,False,179.12,3.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13.0,False,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10.0,False,28.62,1.0


In [21]:
order_items=pd.read_csv(Dataset/"olist_order_items_dataset.csv")
df = pd.merge(df, order_items, on='order_id', how='left')

In [22]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,is_late,payment_value,payment_installments,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,False,38.71,1.0,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,False,141.46,1.0,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,False,179.12,3.0,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13.0,False,72.20,1.0,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10.0,False,28.62,1.0,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


In [23]:
df=pd.merge(df,products,on='product_id',how='left')

In [24]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,is_late,...,price,freight_value,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,False,...,29.99,8.72,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,False,...,118.70,22.76,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,False,...,159.90,19.22,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13.0,False,...,45.00,27.20,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10.0,False,...,19.90,8.72,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery


In [25]:
df=pd.merge(df,sellers,on='seller_id',how='left')

In [26]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,is_late,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,False,...,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,False,...,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,False,...,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13.0,False,...,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10.0,False,...,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP


In [27]:
customers=pd.read_csv(Dataset/"olist_customers_dataset.csv")

In [28]:
df=pd.merge(df,customers, on='customer_id', how='left')

In [29]:
df=pd.merge(df,geo_clean, left_on='customer_zip_code_prefix',right_on='geolocation_zip_code_prefix',how='left')

In [30]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,is_late,...,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,False,...,SP,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,3149.0,-23.576170,-46.587276,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,False,...,SP,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,47813.0,-12.126651,-45.008162,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,False,...,SP,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,75265.0,-16.744472,-48.514624,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13.0,False,...,MG,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,59296.0,-5.774611,-35.273916,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10.0,False,...,SP,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,9195.0,-23.675316,-46.515116,santo andre,SP


In [31]:
df.drop(columns=['geolocation_zip_code_prefix'],inplace=True)

In [32]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,is_late,...,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,False,...,maua,SP,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,-23.576170,-46.587276,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,False,...,belo horizonte,SP,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,-12.126651,-45.008162,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,False,...,guariba,SP,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,-16.744472,-48.514624,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13.0,False,...,belo horizonte,MG,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,-5.774611,-35.273916,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10.0,False,...,mogi das cruzes,SP,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,-23.675316,-46.515116,santo andre,SP


In [33]:
reviews = pd.read_csv(Dataset/'olist_order_reviews_dataset.csv')
reviews_clean = reviews.groupby('order_id').agg({'review_score': 'mean'}).reset_index()

In [34]:
df = pd.merge(df, reviews_clean, on='order_id', how='left')


In [35]:
print(f"Total Orders: {df['order_id'].nunique()}")
print(f"Total Rows: {len(df)}")

Total Orders: 99441
Total Rows: 113425


In [36]:
# Check the range of delays
print(df['delivery_delay_days'].describe())

count    110196.000000
mean        -12.030201
std          10.160157
min        -147.000000
25%         -17.000000
50%         -13.000000
75%          -7.000000
max         188.000000
Name: delivery_delay_days, dtype: float64


In [37]:
print(df[['product_category', 'seller_city', 'payment_value']].isnull().sum())

product_category    2402
seller_city          775
payment_value          3
dtype: int64


In [38]:
df['is_late'] = (df['delivery_delay_days'] > 0).astype(int)

In [39]:
df['revenue_at_risk'] = df['is_late'] * df['payment_value']

In [40]:
import pandas as pd

# 1. Reload the translation file
translation = pd.read_csv('Dataset/product_category_name_translation.csv')

# 2. Check if we actually need to merge
if 'product_category_name_english' not in df.columns:
    print("Merging translation data...")
    # Perform the merge
    df = df.merge(
        translation, 
        left_on='product_category', 
        right_on='product_category_name', 
        how='left'
    )
else:
    print("Column already exists, skipping merge to avoid duplicates.")

# 3. FIX: Check for suffixes (like _x or _y) if merge was run multiple times
for col in df.columns:
    if col.startswith('product_category_name_english'):
        df = df.rename(columns={col: 'product_category_name_english'})
        break # Stop after the first one found

# 4. Handle NaNs and cleanup
df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')

if 'product_category_name' in df.columns:
    df = df.drop(columns=['product_category_name'])

# 5. THE ULTIMATE TEST
print("-" * 30)
if 'product_category_name_english' in df.columns:
    print("✅ SUCCESS: 'product_category_name_english' is in the DataFrame.")
    print("Sample values:", df['product_category_name_english'].unique()[:3])
else:
    print("❌ ERROR: Column still missing. Current columns are:", df.columns.tolist())

Merging translation data...
------------------------------
✅ SUCCESS: 'product_category_name_english' is in the DataFrame.
Sample values: <ArrowStringArray>
['unknown', 'pet_shop', 'cool_stuff']
Length: 3, dtype: str


In [41]:
df.to_csv('olist_master_cleaned.csv', index=False)

In [42]:
print(df.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'is_late', 'payment_value', 'payment_installments', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state', 'review_score', 'revenue_at_risk', 'product_category_name_english']


In [43]:

# 1. Load the raw tables
orders = pd.read_csv('Dataset/olist_orders_dataset.csv')
items = pd.read_csv('Dataset/olist_order_items_dataset.csv')
customers = pd.read_csv('Dataset/olist_customers_dataset.csv')

# 2. Merge to connect State to Freight Value
# Orders -> Customers (for state)
df = orders.merge(customers, on='customer_id')
# Orders -> Items (for freight)
df = df.merge(items, on='order_id')

# 3. Create the State Summary
state_summary = df.groupby('customer_state').agg(
    total_orders=('order_id', 'nunique'),
    avg_freight_value=('freight_value', 'mean')
).reset_index()

# 4. Load your existing summary and merge the freight column in
existing_summary = pd.read_csv('outputs/pbi_state_summary.csv')

# Drop the old summary if it was missing freight and merge
final_state_summary = existing_summary.merge(
    state_summary[['customer_state', 'avg_freight_value']], 
    on='customer_state', 
    how='left'
)

# 5. Overwrite the CSV
final_state_summary.to_csv('outputs/pbi_state_summary.csv', index=False)
print("pbi_state_summary.csv updated with freight data!")

pbi_state_summary.csv updated with freight data!
